## A demonstration of ND filtering in SciPy

First, import necessary stuff

In [3]:
import numpy as np
from scipy.signal import bessel, sosfilt, sosfilt_zi
rng = np.random.default_rng()

Then set up the filters

In [2]:
b = bessel(2, 10, 'low', output='sos', fs=100)

Now make some data to filter

In [6]:
data = []
for _ in range(3):
    data.append(rng.random((100)))
arr_data = np.vstack(data)
print(arr_data.shape)

(3, 100)


Next, we'll filter the data

In [13]:
filt = []
for arr in data:
    filt.append(sosfilt(b, arr))
arr_filt = sosfilt(b, arr_data)
print(np.array_equal(np.vstack(filt), arr_filt))

True


It turns out that we can filter data of arbitrary dimensionality and it won't be an issue. Now we need to look at how to run a single sample through the filter at a time.

In [22]:
# Set up the initial state of the filter
zi = sosfilt_zi(b)
# Just filter everything at one go
y, zf = sosfilt(b, data[0], -1, zi)
# Now filter one sample at a time
y_slow = []
zf = zi
for sample in data[0]:
    yh, zf = sosfilt(b, [sample], -1, zf)
    y_slow.append(yh)
y_slow = np.concat(y_slow)
print(np.array_equal(y, y_slow))

True


So as long as we keep passing the filter delay state in, we can filter one sample at a time. Now for the more complex part--multidimensional filtering. Our first test: does 2D filtering work, filtering one sample of each array at a time?

In [57]:
one_go_filt, _ = sosfilt(b, arr_data, -1, zi.reshape(1, 1, 2).repeat(3, 1))
single_sample_filt = []
for arr in data:
    zf = zi
    filt = []
    for sample in arr:
        yh, zf = sosfilt(b, [sample], -1, zf)
        filt.append(yh)
    single_sample_filt.append(np.concat(filt))
print(np.array_equal(one_go_filt, np.vstack(single_sample_filt)))

True


It does. Now what about filtering one frame with a sample of all arrays at a time?

In [ ]:
single_frame_filt = []
zf = zi.reshape(1, 1, 2).repeat(3, 1)
for i in range(arr_data.shape[-1]):
    yh, zf = sosfilt(b, arr_data[:, i, np.newaxis], 1, zf)
    single_frame_filt.append(yh)
print(np.array_equal(one_go_filt, np.hstack(single_frame_filt)))

[[[ 0.93672024 -0.26206882]
  [ 0.93672024 -0.26206882]
  [ 0.93672024 -0.26206882]]]
True


Now let's scale it up to frames of 2D data. Our pose data will be 33x3 frames.

In [143]:
data3d = rng.random((33, 3, 100))
b = bessel(10, 10, 'low', output='sos', fs=100)
zi = sosfilt_zi(b)
FRAME_SHAPE = (33, 3)

And let's compare our two filter approaches. First, filtering one sample at a time:

In [144]:
single_sample_filt = []
for dim1 in data3d:
    arr = []
    for dim2 in dim1:
        filt = []
        zf = zi
        for sample in dim2:
            y, zf = sosfilt(b, [sample], zi=zf)
            filt.append(y)
        arr.append(np.array(filt).reshape(1, 1, -1))
    single_sample_filt.append(np.concatenate(arr, axis=1))
single_sample_filt = np.concatenate(single_sample_filt, axis=0)
print(single_sample_filt.shape)

(33, 3, 100)


In [145]:
zi.shape

(5, 2)

And second, filtering an entire frame at a time:

In [146]:
single_frame_filt = []
zf = zi.reshape(zi.shape[0], 1, 1, zi.shape[1]).repeat(FRAME_SHAPE[1], 2).repeat(FRAME_SHAPE[0], 1)
for i in range(data3d.shape[-1]):
    yh, zf = sosfilt(b, data3d[:, :, i, np.newaxis], 2, zf)
    single_frame_filt.append(yh)
single_frame_filt = np.concatenate(single_frame_filt, axis=2)

And third, filtering everything in one go:

In [147]:
zf = zi
zf = zi.reshape(zi.shape[0], 1, 1, zi.shape[1]).repeat(FRAME_SHAPE[1], 2).repeat(FRAME_SHAPE[0], 1)
full_filt, _ = sosfilt(b, data3d, zi=zf)

Are they the same?

In [148]:
print(np.array_equal(single_sample_filt, single_frame_filt))
print(np.array_equal(full_filt, single_frame_filt))

True
True


In [149]:
zf.shape

(5, 33, 3, 2)

## FIR 3D filters
First, we test with 1D data

In [7]:
import scipy.signal as signal
import numpy as np
def fir_filt_recursive(coefs, x):
    """
    Recursively filters an array, given filter coefficients.
    :param coefs: The FIR filter coefficients
    :param x: The array to filter
    """
    coefs = coefs[::-1]  # reverse the coefficients
    x = np.concat((np.zeros(coefs.size - 1), x))
    y = []
    for i in range(x.size - coefs.size + 1):
        y.append(np.dot(coefs, x[i:i+coefs.size]))
    return np.array(y)

Does lfilter work properly? Looks like it does.

In [10]:
arr = np.random.random((20))
coefs = np.array([1, -1])
y1 = signal.lfilter(coefs, 1.0, arr)
y2 = fir_filt_recursive(coefs, arr)
print(np.array_equal(y1, y2))

True


What about maintaining delay line state?

In [22]:
zi = signal.lfilter_zi(coefs, 1.0)
y4 = []
zf = zi
for sample in arr:
    y, zf = signal.lfilter(coefs, 1.0, [sample], zi=zf)
    y4.append(y)
y4 = np.concat(y4)
y3, _ = signal.lfilter(coefs, 1.0, arr, zi=zi)
print(np.array_equal(y3, y4))

True


That seems to work too, so let's scale up the dimensions.

In [73]:
data3d = np.random.random((33, 3, 100))
coefs = np.repeat(0.25, 4)
FRAME_SHAPE = (33, 3)

First we do a naive 3D filtering so that we have a benchmark.

In [74]:
def fir_filt_recursive_3d(coefs, x):
    """
    Recursively filters a 3D array, given filter coefficients.
    The third dimension is time.
    :param coefs: The filter coefficients
    :param x: The 3D array to filter
    :return: y
    """
    coefs = coefs[::-1]  # reverse the coefficients
    out = []
    for dim1 in x:
        dim2_y = []
        for dim2 in dim1:
            x_hat = np.concat((np.zeros(coefs.size - 1), dim2))
            y_hat = []
            for i in range(x_hat.size - coefs.size + 1):
                y_hat.append(np.dot(coefs, x_hat[i:i+coefs.size]))
            dim2_y.append(np.array(y_hat).reshape(1, 1, dim2.size))
        out.append(np.concat(dim2_y, 1))
    return np.concat(out, 0)

In [75]:
y1 = fir_filt_recursive_3d(coefs, data3d)
y1.shape

(33, 3, 100)

Now we'll try to set up the same sort of code for batch filtering.

In [76]:
zi = signal.lfilter_zi(coefs, 1.0)
print(zi)
zi = zi.reshape(1, 1, zi.shape[0])
y2 = signal.lfilter(coefs, 1.0, data3d)
print(y2.shape)
np.array_equal(y1, y2)

[0.75 0.5  0.25]
(33, 3, 100)


True

So we know batch filtering should work. The next problem is to figure out how to set up zi.

In [77]:
zi = signal.lfilter_zi(coefs, 1.0)
zi = zi.reshape(1, 1, zi.shape[0])
zi = zi.repeat(FRAME_SHAPE[1], 1)
zi = zi.repeat(FRAME_SHAPE[0], 0)
y3 = signal.lfilter(coefs, 1.0, data3d)
print(y2.shape)
np.array_equal(y1, y3)

(33, 3, 100)


True

Now we just need to figure out how to send individual frames in.

In [88]:
zi = signal.lfilter_zi(coefs, 1.0)
zi = zi.reshape(1, 1, zi.shape[0])
zi = zi.repeat(FRAME_SHAPE[1], 1)
zi = zi.repeat(FRAME_SHAPE[0], 0)
y4 = []
for i in range(data3d.shape[-1]):
    y, zi = signal.lfilter(coefs, 1.0, data3d[:, :, i, np.newaxis], 2, zi=zi)
    y4.append(y)
print(y4[0].shape)
y4 = np.concat(y4, 2)
print(y4.shape)
print(np.array_equal(y3, y4))
print(y3[:2, :2, :2])
print(y4[:2, :2, :2])

(33, 3, 1)
(33, 3, 100)
False
[[[0.03587829 0.04382497]
  [0.11502323 0.25522923]]

 [[0.18385779 0.27143553]
  [0.12068583 0.24869552]]]
[[[0.78587829 0.54382497]
  [0.86502323 0.75522923]]

 [[0.93385779 0.77143553]
  [0.87068583 0.74869552]]]


In [79]:
data3d[:, :, 2, np.newaxis].shape

(33, 3, 1)